# Fine-Tuned FunctionGemma 270M: LiteRT-LM CLI & Web Playground

<table class="tfo-notebook-buttons">
  <td>
    <a target="_blank" href="https://colab.research.google.com/github/google-ai-edge/LiteRT-LM/blob/main/colabs/FunctionGemma_270M_TorchTPU_CLI_and_Web.ipynb">
    <img src="https://www.tensorflow.org/images/colab_logo_32px.png" />
    Run in Google Colab</a>
  </td>
  <td>
    <a target="_blank" href="https://github.com/google-ai-edge/LiteRT-LM/blob/main/colabs/FunctionGemma_270M_TorchTPU_CLI_and_Web.ipynb">
    <img src="https://www.tensorflow.org/images/GitHub-Mark-32px.png" />
    View source on GitHub</a>
  </td>
</table>

[LiteRT-LM](https://github.com/google-ai-edge/LiteRT-LM) is Google’s high-performance edge inference engine that runs LLMs across mobile, web, and desktop with full CPU/GPU/NPU acceleration and production-ready SDKs.

This notebook loads the **Base** and **TorchTPU fine-tuned [FunctionGemma 270M](https://huggingface.co/google/functiongemma-270m-it)** `.litertlm` models and the smart-home tools from the [FunctionGemma TorchTPU Fine-Tuning Colab](https://colab.research.google.com/github/google-ai-edge/LiteRT-LM/blob/main/colabs/FunctionGemma_270M_TorchTPU_Fine_Tuning.ipynb) (Step 8 upload, or the same Colab session), and compares them on the same held-out prompts — `Turn on the living room lights` and `I want the office at 22 degrees` — through:

1. **[LiteRT-LM CLI (`litert-lm`)](https://developers.google.com/edge/litert-lm/cli)**:
   - `litert-lm describe`: Inspect model metadata, modalities (`LlmModelType: function_gemma`), and backend support.
   - `litert-lm run`: Side-by-side tool calling with the smart-home tools (`--preset`), Base vs. Fine-Tuned.
   - `litert-lm benchmark`: Measure prefill and decode throughput on CPU.
2. **[Local OpenAI-Compatible Server (`litert-lm serve`)](https://developers.google.com/edge/litert-lm/cli/openai_server#sending_chat_completion_requests)**:
   - Import both models (`functiongemma-270m-torchtpu` and `functiongemma-270m-base`) and serve `/v1/chat/completions`.
3. **Web Playground & Client-Side WebGPU / WASM ([LiteRT-LM Web Demo](https://google-ai-edge.github.io/LiteRT-LM/web_demos/chat/index.html))**:
   - **In-Notebook Interactive Web UI**: Test custom tool schemas and prompts side-by-side in your browser.
   - **Zero-Code LiteRT-LM Web Chat Demo**: Run the `.litertlm` bundle 100% client-side in Chrome/Edge via WebGPU or WASM.

### Step 1: Install `litert-lm` CLI & Dependencies

In [ ]:
!pip install -q --no-warn-conflicts "litert-lm>=0.1.3" huggingface_hub openai requests
!litert-lm --version

### Step 2: Load the Base & Fine-Tuned `.litertlm` Models and Tools

Set `HF_REPO_ID` / `HF_SUBFOLDER` to where Step 8 of the fine-tuning Colab uploaded the models. If you ran the fine-tuning Colab in this same session, the local files are used directly.

In [ ]:
# @title { display-mode: "form" }
import json
import os
import shutil
from huggingface_hub import hf_hub_download

# Point these at the repo/subfolder you uploaded to in Step 8 of the
# FunctionGemma TorchTPU Fine-Tuning Colab (Base + Fine-Tuned `.litertlm` + `tools.json`).
HF_REPO_ID = "your-hf-username/functiongemma-270m-torchtpu-litertlm"  # @param {type:"string"}
HF_SUBFOLDER = "functiongemma_270m_torchtpu"  # @param {type:"string"}
QUANTIZATION_RECIPE = "dynamic_wi8_afp32"  # @param ["dynamic_wi8_afp32", "dynamic_wi4b32_afp32"]

WORK_DIR = "/content/functiongemma_torchtpu_models"
os.makedirs(WORK_DIR, exist_ok=True)

# Files from a fine-tuning run in this same Colab session are used directly if present.
LOCAL_CANDIDATES = {
    f"functiongemma_270m_torchtpu_{QUANTIZATION_RECIPE}.litertlm": "/content/functiongemma_270m_tpu_finetuned_litertlm/model.litertlm",
    f"functiongemma_270m_base_{QUANTIZATION_RECIPE}.litertlm": "/content/functiongemma_270m_base_litertlm/model.litertlm",
    "tools.json": "/content/sft_data/tools.json",
}


def fetch(filename: str) -> str:
    dst = os.path.join(WORK_DIR, filename)
    if os.path.isfile(dst):
        return dst
    local = LOCAL_CANDIDATES.get(filename)
    if local and os.path.isfile(local):
        shutil.copy2(local, dst)
        return dst
    if HF_REPO_ID.startswith("your-hf-username/"):
        raise ValueError("Set HF_REPO_ID / HF_SUBFOLDER to the repo you uploaded to in Step 8 of the fine-tuning Colab.")
    path_in_repo = f"{HF_SUBFOLDER.strip('/')}/{filename}"
    shutil.copy2(hf_hub_download(repo_id=HF_REPO_ID, filename=path_in_repo), dst)
    return dst


TUNED_LITERTLM_PATH = fetch(f"functiongemma_270m_torchtpu_{QUANTIZATION_RECIPE}.litertlm")
BASE_LITERTLM_PATH = fetch(f"functiongemma_270m_base_{QUANTIZATION_RECIPE}.litertlm")
with open(fetch("tools.json"), encoding="utf-8") as f:
    TOOLS = json.load(f)

# The same held-out prompts shown in the fine-tuning Colab: the base model gets the
# device/zone IDs wrong (or answers in text), the fine-tuned model calls the tool correctly.
DEMO_PROMPTS = ["Turn on the living room lights", "I want the office at 22 degrees"]

for label, path in [("Base FunctionGemma 270M", BASE_LITERTLM_PATH), ("Fine-Tuned FunctionGemma 270M", TUNED_LITERTLM_PATH)]:
    print(f"  - {label}: {path} ({os.path.getsize(path) / (1024 * 1024):.1f} MB)")
print("  - Tools:", ", ".join(t["name"] for t in TOOLS))

### Step 3: LiteRT-LM CLI — Inspect (`describe`), Run (`run`), & Benchmark (`benchmark`)

In [ ]:
import json
import subprocess

# `--preset` loads the same smart-home tools into `litert-lm run`.
PRESET_PATH = "/content/smart_home_tools_preset.py"
with open(PRESET_PATH, "w", encoding="utf-8") as f:
    f.write(
        "import json\n"
        "import litert_lm\n\n"
        "class SmartHomeTool(litert_lm.Tool):\n"
        "    def __init__(self, definition):\n"
        "        self._definition = definition\n"
        "    def get_tool_description(self):\n"
        "        return {'type': 'function', 'function': self._definition}\n"
        "    def execute(self, param):\n"
        "        return {'status': 'ok'}\n\n"
        f"tools = [SmartHomeTool(t) for t in json.loads({json.dumps(json.dumps(TOOLS))})]\n"
    )


def run_cli(cmd: list[str], title: str | None = None, check: bool = False) -> None:
    if title:
        print("=" * 80 + f"\n{title}\n" + "=" * 80)
    res = subprocess.run(cmd, stdin=subprocess.DEVNULL, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, check=check)
    if res.stdout:
        print(res.stdout.rstrip() + "\n")


run_cli(["litert-lm", "describe", TUNED_LITERTLM_PATH], title="Inspect Fine-Tuned .litertlm (`litert-lm describe`)")
for prompt in DEMO_PROMPTS:
    for label, path in [("Base", BASE_LITERTLM_PATH), ("TorchTPU Fine-Tuned", TUNED_LITERTLM_PATH)]:
        run_cli(
            ["litert-lm", "run", path, "--backend=cpu", "--max-num-tokens=512", "--temperature=0.0",
             f"--preset={PRESET_PATH}", f"--prompt={prompt}"],
            title=f"{label} FunctionGemma 270M (`litert-lm run`): {prompt}",
        )
run_cli(["litert-lm", "benchmark", TUNED_LITERTLM_PATH, "--backend=cpu", "--prefill-tokens=64", "--decode-tokens=64", "--max-num-tokens=512"],
        title="Benchmark Fine-Tuned Model (`litert-lm benchmark`)")

### Step 4: Import Models & Start `litert-lm serve` (OpenAI-Compatible REST API)

In [ ]:
# @title { display-mode: "form" }
import json
import os
import subprocess
import time
import requests
from openai import OpenAI

PORT = 9379
run_cli(["litert-lm", "import", TUNED_LITERTLM_PATH, "functiongemma-270m-torchtpu"], check=True)
run_cli(["litert-lm", "import", BASE_LITERTLM_PATH, "functiongemma-270m-base"], check=True)

config_dir = os.path.expanduser("~/.litert-lm")
os.makedirs(config_dir, exist_ok=True)
model_cfg = {"backend": "cpu", "max_num_tokens": 512, "cache": "no", "temperature": 0.0}
with open(os.path.join(config_dir, "config.json"), "w", encoding="utf-8") as f:
    json.dump({"default": model_cfg, "models": {"functiongemma-270m-torchtpu": model_cfg, "functiongemma-270m-base": model_cfg}}, f, indent=2)
run_cli(["litert-lm", "list"])
BASE_URL = f"http://127.0.0.1:{PORT}/v1"


def _is_server_up() -> bool:
    try:
        return requests.get(f"{BASE_URL}/models", timeout=1.5).status_code == 200
    except Exception:
        return False


if not _is_server_up():
    print(f"Starting `litert-lm serve --port {PORT} --cors-origin=*` in background...")
    subprocess.Popen(
        ["litert-lm", "serve", "--host", "127.0.0.1", "--port", str(PORT), "--cors-origin", "*"],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    for _ in range(30):
        if _is_server_up():
            break
        time.sleep(0.5)

assert _is_server_up(), "Failed to start `litert-lm serve`."
print(f"[OK] `litert-lm serve` is listening at {BASE_URL}\n")

OPENAI_TOOLS = [{"type": "function", "function": t} for t in TOOLS]
client = OpenAI(base_url=BASE_URL, api_key="litert-lm")
for prompt in DEMO_PROMPTS:
    print("=" * 80 + f"\nPROMPT: {prompt}")
    for model_id in ("functiongemma-270m-base", "functiongemma-270m-torchtpu"):
        resp = client.chat.completions.create(
            model=model_id,
            messages=[{"role": "user", "content": prompt}],
            tools=OPENAI_TOOLS,
            max_tokens=64,
            temperature=0.0,
        )
        msg = resp.choices[0].message
        calls = [f"{c.function.name}({c.function.arguments})" for c in (msg.tool_calls or [])]
        print(f"  [{model_id}] " + (", ".join(calls) if calls else f"(no tool call) {msg.content!r}"))

### Step 5: Interactive Web Playground (In-Notebook UI Connected to `litert-lm serve`)

In [ ]:
# @title { display-mode: "form" }
import html
from IPython.display import HTML, JSON, display

try:
    from google.colab import output as colab_output
except ImportError:
    colab_output = None


def _query_torchtpu_litert_server(model_id: str, tools_json: str, user_query: str):
    try:
        tools = json.loads(tools_json.strip())
        tools = tools if isinstance(tools, list) else [tools]
        tools_payload = [t if "function" in t else {"type": "function", "function": t} for t in tools]
        r = requests.post(
            f"{BASE_URL}/chat/completions",
            json={
                "model": model_id,
                "messages": [{"role": "user", "content": user_query.strip()}],
                "tools": tools_payload,
                "max_tokens": 64,
                "temperature": 0.0,
            },
            timeout=120,
        )
        r.raise_for_status()
        msg = r.json()["choices"][0]["message"]
        if msg.get("tool_calls"):
            calls = [f"{c['function']['name']}({c['function']['arguments']})" for c in msg["tool_calls"]]
            return JSON({"ok": True, "output": "\n".join(calls)})
        return JSON({"ok": True, "output": "(no tool call) " + (msg.get("content") or "")})
    except Exception as err:
        return JSON({"ok": False, "error": str(err)})


if colab_output is not None:
    colab_output.register_callback("notebook.query_torchtpu_litert_server", _query_torchtpu_litert_server)

PLAYGROUND_DEFAULT_PROMPT = "I want the office at 22 degrees"

demo_buttons = "".join(
    f'<button type="button" class="fg-demo" data-prompt="{html.escape(p)}" style="margin:0 6px 6px 0; padding:5px 10px; border:1px solid #dadce0; border-radius:14px; background:#f8f9fa; cursor:pointer;">{html.escape(p)}</button>'
    for p in DEMO_PROMPTS
)

display(HTML(f"""
<div style="font-family: 'Google Sans', Roboto, sans-serif; max-width: 880px; border: 1px solid #dadce0; border-radius: 12px; padding: 20px; background: #ffffff; color: #202124;">
  <h3 style="margin: 0 0 12px 0; font-size: 18px; color: #1a73e8;">FunctionGemma 270M (TorchTPU Fine-Tuned) — Web Playground</h3>
  <div style="margin-bottom: 12px;">
    <label style="font-size: 13px; font-weight: 600; display: block; margin-bottom: 4px;">Tools (JSON list):</label>
    <textarea id="fg-sys" rows="6" style="width: 100%; box-sizing: border-box; font-family: monospace; font-size: 12px; padding: 8px; border: 1px solid #dadce0; border-radius: 6px;">{html.escape(json.dumps(TOOLS, indent=1))}</textarea>
  </div>
  <div style="margin-bottom: 6px;">
    <label style="font-size: 13px; font-weight: 600; display: block; margin-bottom: 4px;">User Prompt:</label>
    <input id="fg-user" type="text" value="{html.escape(PLAYGROUND_DEFAULT_PROMPT)}" style="width: 100%; box-sizing: border-box; font-size: 14px; padding: 8px 10px; border: 1px solid #dadce0; border-radius: 6px;" />
  </div>
  <div style="margin-bottom: 10px;">{demo_buttons}</div>
  <button id="fg-run-btn" type="button" style="background: #1a73e8; color: white; border: none; padding: 9px 18px; border-radius: 6px; font-weight: 600; cursor: pointer; margin-bottom: 16px;">Run Side-by-Side (Base vs. TorchTPU Fine-Tuned)</button>
  <div style="display: grid; grid-template-columns: 1fr 1fr; gap: 14px;">
    <div style="border: 1px solid #fad2cf; background: #fce8e6; border-radius: 8px; padding: 12px;">
      <div style="font-weight: 600; font-size: 13px; color: #a50e0e; margin-bottom: 6px;">Base FunctionGemma 270M</div>
      <pre id="fg-out-base" style="margin: 0; white-space: pre-wrap; font-family: monospace; font-size: 12px;">Click "Run Side-by-Side" to test...</pre>
    </div>
    <div style="border: 1px solid #ceead6; background: #e6f4ea; border-radius: 8px; padding: 12px;">
      <div style="font-weight: 600; font-size: 13px; color: #0d652d; margin-bottom: 6px;">TorchTPU Fine-Tuned FunctionGemma 270M</div>
      <pre id="fg-out-tuned" style="margin: 0; white-space: pre-wrap; font-family: monospace; font-size: 12px;">Click "Run Side-by-Side" to test...</pre>
    </div>
  </div>
</div>
<script>
(function() {{
  async function callModel(modelId, toolsText, userText) {{
    const raw = await google.colab.kernel.invokeFunction('notebook.query_torchtpu_litert_server', [modelId, toolsText, userText], {{}});
    let parsed = raw && raw.data && raw.data['application/json'];
    if (typeof parsed === 'string') parsed = JSON.parse(parsed);
    return parsed.ok ? parsed.output : ('Error: ' + parsed.error);
  }}
  async function runComparison() {{
    const toolsText = document.getElementById('fg-sys').value;
    const userText = document.getElementById('fg-user').value;
    const baseEl = document.getElementById('fg-out-base');
    const tunedEl = document.getElementById('fg-out-tuned');
    baseEl.textContent = 'Running Base model...';
    tunedEl.textContent = 'Running Fine-Tuned model...';
    baseEl.textContent = await callModel('functiongemma-270m-base', toolsText, userText);
    tunedEl.textContent = await callModel('functiongemma-270m-torchtpu', toolsText, userText);
  }}
  document.querySelectorAll('.fg-demo').forEach(btn => btn.addEventListener('click', () => {{
    document.getElementById('fg-user').value = btn.dataset.prompt;
    runComparison();
  }}));
  document.getElementById('fg-run-btn').addEventListener('click', runComparison);
  runComparison();
}})();
</script>
"""))

### Step 6: Run 100% Client-Side in the Browser (LiteRT-LM Web Chat Demo)

1. Check `DOWNLOAD_WEB_READY_LITERTLM` in the cell below and run it (or download the fine-tuned `.litertlm` from `/content/functiongemma_torchtpu_models/` in the Colab file browser).
2. Open the **[LiteRT-LM Web Chat Demo](https://google-ai-edge.github.io/LiteRT-LM/web_demos/chat/index.html)** in Chrome or Edge.
3. Under **Model Selection**, click **"Upload custom model (.litertlm)..."** and select the fine-tuned `.litertlm` file.
4. If the demo lets you configure tools or a system prompt, paste the contents of `tools.json` there, then try the same prompts:
   - `Turn on the living room lights`
   - `I want the office at 22 degrees`

In [ ]:
# @title { display-mode: "form" }
DOWNLOAD_WEB_READY_LITERTLM = False  # @param {type:"boolean"}

print(f"Web-ready .litertlm model path: {TUNED_LITERTLM_PATH}")
if DOWNLOAD_WEB_READY_LITERTLM:
    from google.colab import files
    files.download(TUNED_LITERTLM_PATH)

## Next Steps: Run Your `.litertlm` Model Everywhere

- **[LiteRT-LM CLI (`litert-lm`)](https://developers.google.com/edge/litert-lm/cli)**: Run, benchmark, and serve `.litertlm` models locally (`pip install litert-lm && litert-lm run model.litertlm`).
- **[LiteRT-LM Web Demo](https://google-ai-edge.github.io/LiteRT-LM/web_demos/chat/index.html)**: Load `.litertlm` models directly in Chrome or Edge with WebGPU.
- **[Google AI Edge Gallery App](https://developers.google.com/edge/gallery)**: Import `.litertlm` models on Android or iOS for interactive on-device inference.